# Limpieza de Datos

# 1. Transformación fechas a signos y eliminar filas
El objetivo es transformar las fechas de nacimiento a signos zodiacales, los datos que no posean la fecha de nacimiento generan el valor desconocido, el cual debe eliminarse al no ser utilizable en el dataset

In [ ]:
import pandas as pd
import os

# Diccionario con las fechas de los signos zodiacales (NO MODIFICAR)
signo_fechas = {
    "Aries": ((3, 21), (4, 19)),
    "Tauro": ((4, 20), (5, 20)),
    "Géminis": ((5, 21), (6, 21)),
    "Cáncer": ((6, 21), (7, 22)),
    "Leo": ((7, 23), (8, 22)),
    "Virgo": ((8, 23), (9, 22)),
    "Libra": ((9, 23), (10, 23)),
    "Escorpio": ((10, 24), (11, 21)),
    "Sagitario": ((11, 22), (12, 21)),
    "Capricornio": ((12, 22), (1, 19)),
    "Acuario": ((1, 20), (2, 18)),
    "Piscis": ((2, 19), (3, 20)),
}

# Función obtener_signo (NO MODIFICAR)
def obtener_signo(dia, mes):
    """Calcula el signo zodiacal a partir del día y mes."""
    for signo, fechas in signo_fechas.items():
        if ((mes == fechas[0][0] and dia >= fechas[0][1]) or
            (mes == fechas[1][0] and dia <= fechas[1][1])):
            return signo
    if (mes == 12 and dia >= 22) or (mes == 1 and dia <= 19):
        return "Capricornio"
    return 'Desconocido'

try:
    # =============================================================================
    # Configuración de rutas
    # =============================================================================
    archivo_original = '../1_data_processed/v0_base_de_datos_unificada.csv'
    archivo_salida = '../1_data_processed/v1_base_filas_validas.csv'
    archivo_eliminadas = '../4_results/v1_filas_eliminadas.csv'

    # Asegurar que exista la carpeta de salida
    os.makedirs(os.path.dirname(archivo_salida), exist_ok=True)

    print("Cargando el archivo completo...")
    df = pd.read_csv(archivo_original, low_memory=False)
    total_filas_original = df.shape[0]
    total_cols_original = df.shape[1]

    # =============================================================================
    # 1. TRANSFORMACIÓN: FECHA -> SIGNO
    # =============================================================================
    print("--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---")

    if 'FECHA_NACIMIENTO' not in df.columns:
        raise KeyError("No existe la columna 'FECHA_NACIMIENTO' en el CSV original.")

    # Convertir FECHA_NACIMIENTO a datetime (inválidos -> NaT)
    df['FECHA_NACIMIENTO'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')

    # Crear SIGNO_ZODIACAL
    df['SIGNO_ZODIACAL'] = df.apply(
        lambda row: obtener_signo(row['FECHA_NACIMIENTO'].day, row['FECHA_NACIMIENTO'].month)
        if pd.notna(row['FECHA_NACIMIENTO']) else 'Desconocido',
        axis=1
    )

    # =============================================================================
    # 2. LIMPIEZA DE FILAS: eliminar filas sin fecha válida o sin signo
    # =============================================================================
    print("--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---")

    mask_invalidas = df['FECHA_NACIMIENTO'].isna() | (df['SIGNO_ZODIACAL'] == 'Desconocido')

    df_eliminadas = df.loc[mask_invalidas].copy()
    df_validas = df.loc[~mask_invalidas].copy()

    print(f"Filas originales: {total_filas_original:,}")
    print(f"Filas eliminadas: {df_eliminadas.shape[0]:,}")
    print(f"Filas válidas: {df_validas.shape[0]:,}")

    # Guardar eliminadas (opcional, pero recomendado)
    if df_eliminadas.shape[0] > 0:
        df_eliminadas.to_csv(archivo_eliminadas, index=False)
        print(f"✅ Filas eliminadas guardadas en: {archivo_eliminadas}")
        
    if "FECHA_NACIMIENTO" in df_validas.columns:
        df_validas.drop(columns=["FECHA_NACIMIENTO"], inplace=True)

    # Guardar dataset válido completo (NO se elimina ninguna columna)
    df_validas.to_csv(archivo_salida, index=False)

    print("\n" + "=" * 60)
    print("¡PROCESO COMPLETADO!")
    print(f"Archivo de salida: {archivo_salida}")
    print(f"Columnas mantenidas: {df_validas.shape[1]} (originales: {total_cols_original})")
    print("=" * 60)

except FileNotFoundError:
    print(f"Error: El archivo '{archivo_original}' no fue encontrado.")
except KeyError as e:
    print(f"Error: {str(e)}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {str(e)}")

Cargando el archivo completo...
--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---
--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---
Filas originales: 5,808,535
Filas eliminadas: 37
Filas válidas: 5,808,498
✅ Filas eliminadas guardadas en: ../1_data_processed/v1_filas_eliminadas.csv

¡PROCESO COMPLETADO!
Archivo de salida: ../1_data_processed/v1_base_filas_validas.csv
Columnas mantenidas: 130 (originales: 130)


# Generar Archivo solo con las columnas necesarias

In [6]:
import pandas as pd
import os

archivo_csv = "../1_data_processed/v1_base_filas_validas.csv"
archivo_csv_slim = "../1_data_processed/v1_base_filas_validas_slim.csv"

os.makedirs("../1_data_processed/", exist_ok=True)

N_DIAG = 35
N_PROC = 30

diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]
proc_cols = [f"PROCEDIMIENTO{i}" for i in range(1, N_PROC + 1)]

cols_needed = ["SIGNO_ZODIACAL", "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols + proc_cols
cols_needed_set = set(cols_needed)

chunksize = 250_000  # baja a 100_000 si aún pesa

# Si existe, lo sobreescribimos
if os.path.exists(archivo_csv_slim):
    os.remove(archivo_csv_slim)

total_rows = 0
first = True

for chunk in pd.read_csv(
    archivo_csv,
    usecols=lambda c: c in cols_needed_set,
    low_memory=False,
    chunksize=chunksize
):
    # Escribe por partes (append). No acumulamos en RAM.
    chunk.to_csv(archivo_csv_slim, mode="a", index=False, header=first)
    first = False

    total_rows += len(chunk)
    print(f"✅ Chunk escrito: {chunk.shape} | acumulado filas: {total_rows:,}")

print("✅ CSV slim guardado en:", archivo_csv_slim)

✅ Chunk escrito: (250000, 69) | acumulado filas: 250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,250,000
✅ C

# 2. Filtro de Columnas
Columnas a mantener asociadas a diagnósticos

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder

# =========================
# CONFIGURACIÓN
# =========================
archivo_entrada = "../1_data_processed/v1_base_filas_validas_slim.csv"

archivo_pre_chi = "../1_data_processed/v2_dataset_pre_chi.csv"
archivo_post_chi = "../1_data_processed/v3_dataset_post_chi.csv"
archivo_resultados_chi = "../4_results/v3_chi2_resultados.csv"

os.makedirs("../1_data_processed/", exist_ok=True)
os.makedirs("../4_results/", exist_ok=True)

TARGET = "SIGNO_ZODIACAL"

SIGNOS_FIJOS = [
    "Acuario", "Aries", "Capricornio", "Cáncer",
    "Escorpio", "Géminis", "Leo", "Libra",
    "Piscis", "Sagitario", "Tauro", "Virgo"
]

N_DIAG = 35
N_PROC = 30

# =========================
# HELPERS
# =========================
def cie10_letra(codigo):
    """Extrae la primera letra A-Z de un código CIE-10."""
    if pd.isna(codigo):
        return "NA"
    s = str(codigo).strip().upper()
    if s == "" or s in {"NAN", "NONE"}:
        return "NA"
    m = re.search(r"[A-Z]", s)
    return m.group(0) if m else "OTROS"

def procedimiento_grupo(codigo):
    """
    Normaliza procedimiento a grupo 00–99 usando el bloque numérico inicial.

    Ejemplos:
      0.66  -> 00
      3.31  -> 03
      01.10 -> 01
      8.2   -> 08
      12    -> 12
    """
    if pd.isna(codigo):
        return "NA"
    s = str(codigo).strip()
    if s == "" or s in {"NAN", "NONE"}:
        return "NA"

    m = re.match(r"(\d+)", s)
    if not m:
        return "OTROS"

    try:
        num = int(m.group(1))
        if 0 <= num <= 99:
            return str(num).zfill(2)
        return "OTROS"
    except Exception:
        return "OTROS"

def normalizar_signo(s):
    """Normaliza el nombre del signo para usarlo como nombre de columna."""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    return s

# =========================
# PASO 0: DEFINIR COLUMNAS Y CARGAR (solo necesarias)
# =========================
diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]
proc_cols = [f"PROCEDIMIENTO{i}" for i in range(1, N_PROC + 1)]

cols_needed = [TARGET, "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols + proc_cols
cols_needed_set = set(cols_needed)

df = pd.read_csv(archivo_entrada, usecols=lambda c: c in cols_needed_set, low_memory=False)
print("Shape inicial:", df.shape)

# Validación mínima
missing = [c for c in ["FECHA_INGRESO", "FECHAALTA", TARGET, "ESPECIALIDAD_MEDICA"] if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas mínimas en el CSV slim: {missing}")

# =========================
# PASO 1: ESTANCIA_DIAS
# =========================
df["FECHA_INGRESO"] = pd.to_datetime(df["FECHA_INGRESO"], errors="coerce")
df["FECHAALTA"] = pd.to_datetime(df["FECHAALTA"], errors="coerce")

df["ESTANCIA_DIAS"] = (df["FECHAALTA"] - df["FECHA_INGRESO"]).dt.days
df["ESTANCIA_DIAS"] = df["ESTANCIA_DIAS"].fillna(0).astype(np.int32)
df.loc[df["ESTANCIA_DIAS"] < 0, "ESTANCIA_DIAS"] = 0

# Liberar fechas ASAP
df.drop(columns=["FECHA_INGRESO", "FECHAALTA"], inplace=True)

# =========================
# PASO 2: FILTRO BASE
# =========================
df = df[[TARGET, "ESPECIALIDAD_MEDICA", "ESTANCIA_DIAS"] + diag_cols + proc_cols].copy()
print("Shape después del filtro base:", df.shape)

# =========================
# PASO 3: TRANSFORMACIONES
# =========================
# Diagnósticos -> letra
for c in diag_cols:
    df[c] = df[c].apply(cie10_letra)

# Procedimientos -> grupo 00-99
for c in proc_cols:
    df[c] = df[c].apply(procedimiento_grupo)

print("✅ Transformación diagnósticos/procedimientos completada.")

# =========================
# PASO 4: ONE-HOT (solo categóricas)
# (ahorro RAM: convertir a category antes de get_dummies)
# =========================
cat_cols = ["ESPECIALIDAD_MEDICA"] + diag_cols + proc_cols
for c in cat_cols:
    df[c] = df[c].astype("category")

# Si tu RAM vuelve a sufrir, cambia a: sparse=True
X_cat = pd.get_dummies(df[cat_cols], drop_first=False)
print("✅ One-hot listo:", X_cat.shape)

# =========================
# PASO 5: CHI² (categóricas one-hot)
# =========================
le = LabelEncoder()
y_encoded = le.fit_transform(df[TARGET].astype("object"))

chi_scores, p_values = chi2(X_cat, y_encoded)

resultados = pd.DataFrame({
    "variable": X_cat.columns,
    "chi2": chi_scores,
    "p_value": p_values
}).sort_values("chi2", ascending=False)

resultados.to_csv(archivo_resultados_chi, index=False)
print("✅ Resultados chi guardados en:", archivo_resultados_chi)

# =========================
# PASO 6: BINARIZAR TARGET (ONE-VS-REST)
# =========================
Y_bin = pd.DataFrame(index=df.index)
for signo in SIGNOS_FIJOS:
    colname = f"signo_zodiacal_{normalizar_signo(signo)}"
    Y_bin[colname] = (df[TARGET] == signo).astype(np.uint8)

print("✅ Targets binarios creados:", list(Y_bin.columns))

# =========================
# GUARDAR PRE_CHI (X_cat + continua + targets binarios)
# =========================
df_pre_chi = pd.concat([X_cat, df["ESTANCIA_DIAS"], Y_bin], axis=1)
df_pre_chi.to_csv(archivo_pre_chi, index=False)
print("✅ Dataset PRE_CHI guardado:", df_pre_chi.shape, "->", archivo_pre_chi)

# =========================
# FILTRAR VARIABLES SIGNIFICATIVAS (p < 0.05) Y GUARDAR POST_CHI
# =========================
vars_significativas = resultados.loc[resultados["p_value"] < 0.05, "variable"].tolist()
X_post = X_cat[vars_significativas]

df_post_chi = pd.concat([X_post, df["ESTANCIA_DIAS"], Y_bin], axis=1)
df_post_chi.to_csv(archivo_post_chi, index=False)
print("✅ Dataset POST_CHI guardado:", df_post_chi.shape, "->", archivo_post_chi)

Shape inicial: (5808498, 39)
✅ Transformación diagnósticos completada.
✅ One-hot listo: (5808498, 1044)


MemoryError: Unable to allocate 1.17 GiB for an array with shape (27, 5808498) and data type float64

In [22]:
for col in df_pre_chi.columns:
    print(col)

ESPECIALIDAD_MEDICA_ADOLESCENCIA (PEDIATRIA)
ESPECIALIDAD_MEDICA_ANATOMÍA PATOLÓGICA
ESPECIALIDAD_MEDICA_ANESTESIOLOGÍA
ESPECIALIDAD_MEDICA_CARDIOLOGÍA
ESPECIALIDAD_MEDICA_CARDIOLOGÍA PEDIÁTRICA
ESPECIALIDAD_MEDICA_CIRUGÍA CABEZA CUELLO Y PLÁSTICA MAXILO FACIAL
ESPECIALIDAD_MEDICA_CIRUGÍA CARDIOVASCULAR
ESPECIALIDAD_MEDICA_CIRUGÍA COLOPROCTOLÓGICA
ESPECIALIDAD_MEDICA_CIRUGÍA DE CABEZA, CUELLO Y MAXILOFACIAL
ESPECIALIDAD_MEDICA_CIRUGÍA DE TÓRAX
ESPECIALIDAD_MEDICA_CIRUGÍA DIGESTIVA
ESPECIALIDAD_MEDICA_CIRUGÍA GENERAL
ESPECIALIDAD_MEDICA_CIRUGÍA PEDIÁTRICA
ESPECIALIDAD_MEDICA_CIRUGÍA PLÁSTICA Y REPARADORA
ESPECIALIDAD_MEDICA_CIRUGÍA TÓRAX
ESPECIALIDAD_MEDICA_CIRUGÍA VASCULAR PERIFÉRICA
ESPECIALIDAD_MEDICA_CIRUGÍA Y TRAUMATOLOGÍA BUCO MAXILOFACIAL
ESPECIALIDAD_MEDICA_CIRUGÍA Y TRAUMATOLOGÍA BUCOMAXILOFACIAL
ESPECIALIDAD_MEDICA_CIRUJANO DENTISTA
ESPECIALIDAD_MEDICA_COLOPROCTOLOGÍA
ESPECIALIDAD_MEDICA_CUIDADOS INTENSIVOS PEDIÁTRICO
ESPECIALIDAD_MEDICA_DERMATOLOGÍA
ESPECIALIDAD_MEDICA_DERMAT